# B4: Named Entity Recognition (NER) and Evaluation

Preprocess the text, apply NER with (i) a rule-based approach (pattern matching) and (ii) a pre-trained statistical model (spaCy), extract and classify entities, evaluate with precision, recall and F1-score, and compare the two approaches.

**Dataset:** CoNLL-2003 English (Reuters news, labels PER, LOC, ORG, MISC) and a few custom news sentences

In [2]:
# spacy : the statistical NER model and the EntityRuler for rules
!pip install -q spacy pandas
!python -m spacy download en_core_web_sm

In [3]:
import urllib.request
from collections import Counter, defaultdict
import pandas as pd
import spacy
from spacy.tokens import Doc

## (a) Load and preprocess CoNLL-2003

Each line is `word POS chunk NER-tag`; a blank line ends a sentence; `-DOCSTART-` separates articles.

In [4]:
URL = "https://raw.githubusercontent.com/glample/tagger/master/dataset/"
raw = urllib.request.urlopen(URL + "eng.testb").read().decode("utf-8")
print(raw[:300])

SOCCER NN I-NP O
- : O O
JAPAN NNP I-NP I-LOC
GET VB I-VP O
LUCKY NNP I-NP O
WIN NNP I-NP O
, , O O
CHINA NNP I-NP I-PER
IN IN I-PP O
SURPRISE DT I-NP O
DEFEAT NN I-NP O
. . O O

Nadim NNP I-NP I-PER
Ladki NNP I-NP I-PER

AL-AIN NNP I-NP I-LOC
, , O O
United NNP I-NP I-LOC
Arab NNP I-NP I-LOC
Emirat


In [5]:
def read_conll(name):
    text = urllib.request.urlopen(URL + name).read().decode("utf-8")
    sents, words, tags = [], [], []
    for line in text.splitlines():
        if not line.strip():                      # blank line: sentence finished
            if words: sents.append((words, tags))
            words, tags = [], []
        elif not line.startswith("-DOCSTART-"):
            parts = line.split()
            words.append(parts[0]); tags.append(parts[-1])
    return sents

train = read_conll("eng.train")
test = read_conll("eng.testb")
print(len(train), "train sentences,", len(test), "test sentences")
print(list(zip(*test[1])))

14041 train sentences, 3453 test sentences
[('Nadim', 'I-PER'), ('Ladki', 'I-PER')]


In [6]:
# turn the per-word tags into entity spans (start, end, type)
def spans(tags):
    out, start, typ = [], None, None
    for i, t in enumerate(tags + ["O"]):
        if t == "O" or t.startswith("B-") or t[2:] != typ:     # the current entity ends here
            if start is not None: out.append((start, i, typ))
            start, typ = (i, t[2:]) if t != "O" else (None, None)
    return out

words, tags = test[1]
[(" ".join(words[s:e]), t) for s, e, t in spans(tags)]

[('Nadim Ladki', 'PER')]

In [7]:
Counter(t for _, tags in test for _, _, t in spans(tags))

Counter({'LOC': 1668, 'ORG': 1661, 'PER': 1617, 'MISC': 702})

## (b-i) Rule-based NER with pattern matching

spaCy's `EntityRuler` matches patterns:
1. a list of names (gazetteer) taken from the training data,
2. a few token patterns: a title such as "Mr." before a capitalised word means a person; a capitalised word followed by "Inc", "Bank", "Party" ... means an organisation; any other two capitalised words in a row are guessed to be a person.

When patterns overlap, the longest match wins, so a known name from the list beats the two-word guess.

In [8]:
# gazetteer: every entity seen in training, with its most frequent label
seen = defaultdict(Counter)
for words, tags in train:
    for s, e, t in spans(tags):
        seen[" ".join(words[s:e])][t] += 1
from spacy.lang.en.stop_words import STOP_WORDS
gazetteer = {name: c.most_common(1)[0][0] for name, c in seen.items()
             if name.lower() not in STOP_WORDS}          # skip words like "The" that were once part of a name
print(len(gazetteer), "names, e.g.", list(gazetteer.items())[:6])

8073 names, e.g. [('EU', 'ORG'), ('German', 'MISC'), ('British', 'MISC'), ('Peter Blackburn', 'PER'), ('BRUSSELS', 'LOC'), ('European Commission', 'ORG')]


In [9]:
patterns = [{"label": label, "pattern": name} for name, label in gazetteer.items()]
patterns += [
    {"label": "PER", "pattern": [{"TEXT": {"IN": ["Mr.", "Mrs.", "Ms.", "Dr.", "President"]}}, {"IS_TITLE": True, "OP": "+"}]},
    {"label": "ORG", "pattern": [{"IS_TITLE": True, "OP": "+"}, {"TEXT": {"IN": ["Inc", "Corp", "Ltd", "Bank", "Party", "Club", "United", "University"]}}]},
    {"label": "PER", "pattern": [{"IS_TITLE": True, "IS_ALPHA": True}, {"IS_TITLE": True, "IS_ALPHA": True}]},   # two capitalised words: guess a person
]
rule_nlp = spacy.blank("en")
ruler = rule_nlp.add_pipe("entity_ruler")
ruler.add_patterns(patterns)
print(len(patterns), "patterns")

8076 patterns


## (b-ii) Pre-trained statistical model: spaCy

In [10]:
nlp = spacy.load("en_core_web_sm")

# spaCy uses 18 labels; map them to CoNLL's 4 (dates, numbers, money are not entities in CoNLL)
to_conll = {"PERSON": "PER", "ORG": "ORG", "GPE": "LOC", "LOC": "LOC", "FAC": "LOC",
            "NORP": "MISC", "EVENT": "MISC", "PRODUCT": "MISC", "WORK_OF_ART": "MISC", "LAW": "MISC", "LANGUAGE": "MISC"}

## (c) Extract and classify entities

In [11]:
sentence = "Mr. Sundar Pichai met officials of Tata Motors Ltd and the Reserve Bank in Mumbai on Monday."
print("rules:", [(e.text, e.label_) for e in rule_nlp(sentence).ents])
print("spaCy:", [(e.text, e.label_, to_conll.get(e.label_, "-")) for e in nlp(sentence).ents])

rules: [('Mr. Sundar Pichai', 'PER'), ('Tata Motors Ltd', 'ORG'), ('Reserve Bank', 'PER')]
spaCy: [('Sundar Pichai', 'PERSON', 'PER'), ('Tata Motors Ltd', 'ORG', 'ORG'), ('the Reserve Bank', 'ORG', 'ORG'), ('Mumbai', 'GPE', 'LOC'), ('Monday', 'DATE', '-')]


In [12]:
# run both on the CoNLL test sentences, keeping CoNLL's own tokens
def predict(model, sents, label_map=None):
    out = []
    for doc in model.pipe(Doc(model.vocab, words=w) for w, _ in sents):
        ents = []
        for e in doc.ents:
            label = label_map.get(e.label_) if label_map else e.label_
            if label: ents.append((e.start, e.end, label))
        out.append(ents)
    return out

rule_pred = predict(rule_nlp, test)
spacy_pred = predict(nlp, test, to_conll)
gold = [spans(t) for _, t in test]

words = test[1][0]
print("gold :", [(" ".join(words[s:e]), t) for s, e, t in gold[1]])
print("rules:", [(" ".join(words[s:e]), t) for s, e, t in rule_pred[1]])
print("spaCy:", [(" ".join(words[s:e]), t) for s, e, t in spacy_pred[1]])

gold : [('Nadim Ladki', 'PER')]
rules: [('Nadim Ladki', 'PER')]
spaCy: [('Nadim Ladki', 'PER')]


## (d) Precision, recall and F1-score

An entity counts as correct only if its span **and** its type match the gold entity exactly.

In [13]:
def evaluate(gold, pred):
    rows = []
    for typ in ["PER", "LOC", "ORG", "MISC", "ALL"]:
        g = {(i, *x) for i, ents in enumerate(gold) for x in ents if typ == "ALL" or x[2] == typ}
        p = {(i, *x) for i, ents in enumerate(pred) for x in ents if typ == "ALL" or x[2] == typ}
        tp = len(g & p)
        P = tp / len(p) if p else 0
        R = tp / len(g) if g else 0
        rows.append({"type": typ, "precision": P, "recall": R, "f1": 2 * P * R / (P + R) if P + R else 0})
    return pd.DataFrame(rows).set_index("type").round(3)

pd.concat({"rule-based": evaluate(gold, rule_pred), "spaCy": evaluate(gold, spacy_pred)}, axis=1)

rule-based                   spaCy              
      precision recall     f1 precision recall     f1
type                                                 
PER       0.511  0.610  0.556     0.716  0.588  0.646
LOC       0.899  0.746  0.816     0.770  0.730  0.749
ORG       0.747  0.468  0.576     0.430  0.310  0.360
MISC      0.847  0.530  0.652     0.675  0.564  0.614
ALL       0.705  0.599  0.648     0.656  0.545  0.595

## Custom news sentences

In [14]:
custom = [
    ("Infosys was founded in Pune by Narayana Murthy .", [("Infosys", "ORG"), ("Pune", "LOC"), ("Narayana Murthy", "PER")]),
    ("Lionel Messi led Argentina to victory over France in Qatar .", [("Lionel Messi", "PER"), ("Argentina", "LOC"), ("France", "LOC"), ("Qatar", "LOC")]),
    ("Tata Motors bought Jaguar Land Rover from Ford .", [("Tata Motors", "ORG"), ("Jaguar Land Rover", "ORG"), ("Ford", "ORG")]),
    ("The Reserve Bank of India is based in Mumbai .", [("Reserve Bank of India", "ORG"), ("Mumbai", "LOC")]),
    ("Virat Kohli scored a century against Australia in Perth .", [("Virat Kohli", "PER"), ("Australia", "LOC"), ("Perth", "LOC")])]

def text_ents(model, text, label_map=None):
    doc = model(Doc(model.vocab, words=text.split()))
    return {(e.text, label_map.get(e.label_) if label_map else e.label_) for e in doc.ents} - {(e.text, None) for e in doc.ents}

rows = []
for name, model, m in [("rule-based", rule_nlp, None), ("spaCy", nlp, to_conll)]:
    tp = n_pred = n_gold = 0
    for text, g in custom:
        p = text_ents(model, text, m)
        tp += len(p & set(g)); n_pred += len(p); n_gold += len(g)
    P, R = tp / n_pred, tp / n_gold
    rows.append({"model": name, "precision": P, "recall": R, "f1": 2 * P * R / (P + R)})
pd.DataFrame(rows).set_index("model").round(3)

,precision,recall,f1
model,,,
rule-based,0.818,0.600,0.692
spaCy,0.933,0.933,0.933


In [15]:
for text, g in custom:
    print(text)
    print("   rules:", text_ents(rule_nlp, text))
    print("   spaCy:", text_ents(nlp, text, to_conll))

Infosys was founded in Pune by Narayana Murthy .
   rules: {('Narayana Murthy', 'PER')}
   spaCy: {('Narayana Murthy', 'PER'), ('Infosys', 'ORG'), ('Pune', 'LOC')}
Lionel Messi led Argentina to victory over France in Qatar .
   rules: {('Argentina', 'LOC'), ('Lionel Messi', 'PER'), ('France', 'LOC')}
   spaCy: {('Argentina', 'LOC'), ('Lionel Messi', 'PER'), ('Qatar', 'LOC'), ('France', 'LOC')}
Tata Motors bought Jaguar Land Rover from Ford .
   rules: {('Ford', 'ORG'), ('Tata Motors', 'PER'), ('Jaguar Land', 'PER')}
   spaCy: {('Ford', 'ORG'), ('Tata Motors', 'ORG'), ('Jaguar Land Rover', 'ORG')}
The Reserve Bank of India is based in Mumbai .
   rules: {('Reserve Bank of India', 'ORG')}
   spaCy: {('Mumbai', 'LOC'), ('The Reserve Bank of India', 'ORG')}
Virat Kohli scored a century against Australia in Perth .
   rules: {('Virat Kohli', 'PER'), ('Perth', 'LOC'), ('Australia', 'LOC')}
   spaCy: {('Virat Kohli', 'PER'), ('Perth', 'LOC'), ('Australia', 'LOC')}


## (e) Compare and justify
- On **CoNLL** the rule-based system is helped by its gazetteer, which comes from the CoNLL training file: the test articles are from the same news source and year, so many names repeat. spaCy never saw CoNLL and uses different label definitions (for example sports teams named after countries are ORG in CoNLL but GPE in spaCy).
- On **new text** (the custom sentences) the gazetteer has few of the names, so the rules miss entities or guess the wrong type ("Tata Motors" becomes a person by the two-word rule); spaCy uses context and word shape and generalises much better.
- Rules: transparent, precise for known names, no training needed, but brittle and need constant updating.
- Statistical model: handles unseen names, but its mistakes are harder to explain and fix, and its label set may not match yours.